<img src="pictures/hanze.png" align="right" />

# <span id="0">Computer-exam BFVM23DATASC5 - Supervised Learning</span>

## Study year 2025-2026 - first opportunity

### Fri. 14 Nov 2025, 12:30-16:30, ZP11/H1.86

Take a minute to read the following instructions and information.

## Instructions

This is an "open-book" exam that you take on school workstations using your own account. You are allowed to use any digital materials in your home folder, look up information on the internet, as well as consult any written materials. However, you are **not** allowed to utilize your own devices (e.g. telephone or laptop) or use communication media or AI-tools (e.g. chat and ChatGPT; search engines are permitted). You are allowed to leave the exam room for toilet or coffee breaks, except during the first and last hour of the scheduled exam time. You are not permitted to confer with fellow students about concrete code.

<div class="alert alert-warning">
    
# ⚠️

**Note:** Your editor may auto-complete existing variable names or show a pop-up with function arguments or built-in help, but it may not suggest additional fragments of code or expand comments. Turn off any "AI plugins" that your IDE might use: disable all plugins under "Local AI/ML Tools" in PyCharm Settings > Plugins, in VS-Code disable AI-powered Extensions like Copilot or Chat, and uninstall `jupyter-ai` if that has been installed as a notebook extension. Violations will be reported as fraud!

</div>

This exam consists of the following assignments:

1. **<a href="#1">Part I</a>** <small>[20 pts]</small>

2. **<a href="#2">Part II</a>** <small>[20 pts]</small>

3. **<a href="#3">Part III</a>** <small>[20 pts]</small>

4. **<a href="#4">Part IV</a>** <small>[30 pts]</small>

Each part directs you to perform a particular supervised-machine learning analysis. If you do not understand what is meant, you may ask the exam's supervisor for clarification. Execute the analysis that is required. Use your own judgement to perform any steps that you deem necessary, even if those are not all explicitly requested.

The various parts have a logical order, but can be answered separately; therefore, you can always continue to a next part. If you are unable to (fully) solve one part, an explanation of your intended solution in the form of text or pseudo-code may be awarded partial credit.

<div class="alert alert-info">
    
# ℹ️

**Note:** Do not only provide code solutions that show *what* you do, but also explain by means of comments and/or text markup *why* you perform steps, as well as what you *conclude* from the output of an analysis!

</div>

Save your notebook regularly to avoid inadvertent loss of your progress! Instructions on how to submit your answers after you finish are provided at the very <a href="#X">bottom</a>.

## Assessment

Your grade will be calculated as follows:

$$
\text{Grade} = 1 + 9 \cdot \frac {\text{Points Scored}} {\text{Maximum Score}}
$$

All parts have the possible number of points to be scored indicated.

* 40% of your grade is based on the insightfulness of provided explanations, motivations, interpretations, and conclusions drawn from your analyses in **text**.

* 40% of your grade is based on the correctness, completeness, efficiency, and intelligibility of the python **code** that you submit.

* 20% of your grade is based on the implementation and use of machine learning functions or classes in a **module** that you already created previously in the course of the lessons.

If deemed necessary, you may be invited to an oral interview after the exam to provide further explanations. This interview, as well as your observed activities during the exam session itself can be taken into account in the assessment of the above aspects.

***

## Data

On BrightSpace you will find a zip-file containing the following materials:

* this exam `BFVM23DATASC5_T_DataScience5_SupervisedLearning_2526_DSLS_KRPE-LADR.ipynb` (also as `*.html`);
* a `data` folder with data files;
* a `pictures` folder with illustrations.

In the following assignments you will be working with a dataset from a study that was designed to analyze the usefulness of [Cardiotocograms](https://en.wikipedia.org/wiki/Cardiotocography) (CTGs) as a simple and cost accessible option to assess fetal health, allowing healthcare professionals to take action in order to prevent child and maternal mortality. The equipment itself works by sending ultrasound pulses and reading its response, thus shedding light on fetal heart rate (FHR), uterine contractions (UC), and more.

<img src="pictures/612984125-H-768x525.jpg" width="25%" />

Run the cell below to import the data into a variable `data` and show a sample of the 2126 instances.

In [ ]:
import pandas as pd

data = pd.read_csv(
    '../test_exam/BFVM23DATASC5_T_DataScience5_SupervisedLearning_2526_DSLS_KRPE-LADR/data/fetal_health_dataset.csv'). \
       apply(lambda col: col.astype('category') if col.dtypes == 'object' else col)

data

Key information about the interpretation of the variables is summarized in a codebook that is also provided as the readable file `'fetal_health_codebook.csv'`. If necessary, some further background information on the dataset and can be found in the original [online source](https://doi.org/10.24432/C51S4N).

Below, a succinct overview is shown of all 23 **features** in the above dataframe, accompanied by a brief **description** from the codebook, while also showing their total number of **non-available values** as well as their **data type**. Moreover, for numeric features the **mean ± SD** and **min-max range** are given, whereas for categorical features the **distinct labels** and **number of levels** of the variable are shown.

In [ ]:
codebook = pd.read_csv(
    '../test_exam/BFVM23DATASC5_T_DataScience5_SupervisedLearning_2526_DSLS_KRPE-LADR/data/fetal_health_codebook.csv',
    index_col='feature', names=['feature', 'description'])
codebook = codebook.assign(NAs=data.isna().sum(), dtype=data.dtypes, values='')
for feature in data.select_dtypes('number'):
    codebook.loc[feature, 'values'] = f'{data[feature].mean():.2f} ± {data[feature].std():.2f} [{data[feature].min():.2f} - {data[feature].max():.2f}]'
for feature in data.select_dtypes('category'):
    codebook.loc[feature, 'values'] = '|'.join(data[feature].cat.categories) + f' [{len(data[feature].cat.categories)} levels]'

codebook

Have a good look at the above overview. Note that there are no missing values. Most features are numeric, in the form of floating-point values or integers. Two features contain categorical labels, one with 10 levels and one with 3.

For your convenience, the distribution of the various data columns and their pairwise correlations are plotted below. You are not required to perform an exploratory data analysis, but — if necessary — run any additional code that you deem useful to get insight into the nature of the available data.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

plt.figure(figsize=(16, 24))
for index, feature in enumerate(data, start=1):
    plt.subplot(6, 4, index)
    plt.title(feature)
    if data[feature].dtype == 'category':
        value_counts = data[feature].value_counts(sort=False)
        plt.pie(value_counts, labels=value_counts.index, startangle=90, counterclock=False)
    else:
        plt.hist(data[feature], bins='doane', rwidth=0.8)
plt.show()

In [ ]:
correlations = data.corr(method='spearman', numeric_only=True)

plt.figure(figsize=(8, 6))
plt.imshow(correlations, cmap='RdYlBu', vmin=-1.0, vmax=1.0)
plt.xticks(range(len(correlations)), labels=correlations.columns, rotation=60, ha="right", rotation_mode="anchor")
plt.yticks(range(len(correlations)), labels=correlations.columns)
plt.title('Correlation matrix (Spearman $R$)')
plt.colorbar(shrink=0.4)
plt.show()

The CTGs of pregnant women were classified by three experts who were specialized in obstetrics, with their interpretation being considered to be the gold standard. The last data column with the Fetal state class code (NSP) indicates whether a foetus is diagnosed to be *normal* (N), *suspect* (S) or *pathologic* (P). This feature will serve as an *ordinal* target for a predictive model, since the *suspect* diagnosis is intermediate between the *normal* and *pathologic* diagnoses.

In the following assignments, you will explore a number of ways to deal with the prediction of such an ordinal target. Moreover, you will compare the results of your own models to some published work.

<a id="1" href="#0" style="text-align: right; display: block;">Back to top</a>

## ❶ Part I: Ordinal prediction by means of regression <small>[20 pts]</small>

A first method to predict ordinal labels is to re-encode the target as a numeric variable. In this case, the labels ${\text{N},\text{S},\text{P}}$ may for instance be re-encoded as ${-1,0,+1}$ (in order of increasing certainty of a positive diagnosis). Then, the target becomes numeric and the problem reduces to regression.

When making a novel prediction using regression, a numeric value is obtained that may not exactly correspond with a valid label. However, this can be solved by rounding to the nearest valid output. For example, given the previous re-encoding, an output of the model equal to $\hat{y} = -0.73$ may be rounded to $\hat{y} = -1$, corresponding with a *normal* (N) diagnosis.

**Assignment**

* Use a standard linear regression model to try and predict the Fetal state class code (NSP) from *all* 22 other features. Explain what preprocessing you perform on the features, if any, and why.

* Based on resubstitution without cross-validation, plot the predicted Fetal state class code (NSP) against the true Fetal state class code, and compute (at least) two different measures of the performance of the model. Motivate why you picked these measures; in your opinion, does the model perform well?

* List the top features that seem to be the most important factors that influence the NSP diagnosis according to the fitted model. How do you interpret these findings?

In [ ]:
# YOUR ANSWER GOES HERE
# First, if the exercise didn't explicitly state to use all 22 features then I'd consider removing the mean, mode, variance, all the histogram stuff.
# It seems to me that this data is already inside other variables and using it may lead to there being an unneeded focus on the feature that this values were
# retrieved from.
# Second, scaling. I can already see a question about 'top features', if we don't scale just looking at the coeffecients. If we do scale, the coefficients might
# tell us exciting information. Strictly speaking, linear regression doesn't really need scaling, but I imagine it might make gradient descent easier when all
# the data is on one scale. We settle on scaling beforehand. Additioanly, one hot encoding for the categorical value.
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from model import LinearRegression
X, y = data.loc[:, data.columns != "NSP"], data["NSP"]
X = X.loc[:, X.columns != "CLASS"]

def ordinal(y):
    result = []
    for item in y:
        if item == "N":
            result.append(-1)
        if item == "S":
            result.append(0)
        if item == "P":
            result.append(1)
    return result

y_ordinal = ordinal(y)
enc = OneHotEncoder(handle_unknown='ignore', drop="first")
class_one_hot = enc.fit_transform(pd.DataFrame(data["CLASS"]))
scaler = StandardScaler()
X_scaled= scaler.fit_transform(X)
combined = np.hstack([X_scaled,class_one_hot.toarray()])

model = LinearRegression().fit(combined, y_ordinal)
predictions = model.predict(combined)

In [ ]:
refs = np.array([-1, 0, 1])
labels = np.array(['N', 'S', 'P'])
idx = np.abs(predictions[:, None] - refs[None, :]).argmin(axis=1)
result = labels[idx]
print(refs[idx])
print(result)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, mean_absolute_error

plt.scatter(x=y_ordinal,y=predictions, color='red')
plt.axhline(y=0.5, color='black')
plt.axhline(y=-.5, color='black')
plt.show()

print(confusion_matrix(y, result, labels=["N", "S", "P"]))
print("MEA:", mean_absolute_error(y_ordinal, predictions))
print("F1 score:", f1_score(y, result, average='macro'))

In [ ]:
sorted_coefs = np.argsort(abs(model.coef_))
print(len(model.coef_))
print(combined.shape)
additional_columns = [f"class_{i}" for i in range(1, 10)]
columns = list(X.columns) + additional_columns
print(np.asarray(columns)[sorted_coefs])

<a id="2" href="#0" style="text-align: right; display: block;">Back to top</a>

## ❷ Part II: Ordinal prediction by means of multinomial classification <small>[20 pts]</small>

A second way to deal with ordinal targets is to simply *ignore* the natural ordering of the labels. In this particular case, the labels N/S/P are assumed to correspond with three "unrelated" classes. Then, the target becomes nominal and the problem reduces to (multinomial) classification.

Here we will consider three potentially suitable classification methods that are all based on similar ideas, involving Bayesian prediction of class membership probabilities. They only differ in the assumptions that they place on the features:

1. Gaussian Naive Bayes (GNB)
2. Linear Discriminant Analysis (LDA)
3. Quadratic Discriminant Analysis (QDA)

Below, you aim to predict the Fetal state class code (NSP) from only a selection of the available features. Not all of the above methods are equally suited for that purpose.

**Assignment**

* Manually select precisely 10 features that you deem most suitable to use in your model based on their histogram distributions and/or pairwise correlations (as shown in the Data section above). Briefly explain on what basis you chose these features.

* Split these 10 features as well as the Fetal state class code (NSP) target into a training and test dataset of *equal* size. Employ stratification.

* Out of the three types of models listed above (i.e. GNB, LDA, and QDA), fit the one that you deem most appropriate to the training dataset. Mention some relevant differences between the three, and explain why you picked this model.

* Determine the model's average F1-score for the training dataset as well as the test dataset. Explain if and how you can assess whether the model seems to be underfitting and/or overfitting.

In [ ]:
from sklearn.model_selection import train_test_split
# YOUR ANSWER GOES HERE
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score, confusion_matrix, mean_absolute_error
#These are not the appropriate values.
X = data[["MSTV", "Width", "AC", "LB", "FM", "UC", "Nmax","DS", "DL", "Tendency"]]
y = data["NSP"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

clf = GaussianNB()
clf.fit(X_train, y_train)
predictions = clf.predict(X_test)
print(confusion_matrix(y_test, predictions))
print("F1 score:", f1_score(y_test, predictions, average='macro'))

<a id="3" href="#0" style="text-align: right; display: block;">Back to top</a>

## ❸ Part III: Ordinal prediction by means of an ensemble model <small>[20 pts]</small>

A third way to deal with ordinal targets is to use an *ensemble* of binary classifiers. The following approach is comparable to *(hard) voting*, in particular to the *one-vs-one* or *one-vs-rest* meta-classifiers that combine multiple binary base-classifiers to achieve a single multiclass classification.

In order to achieve ordinal class prediction, we train multiple binary base-classifiers to distinguish between sets of "low" class labels against sets of "high" class labels. If we have a total of $C$ different ordinal classes that we shall indicate with ordered labels ${0,1,\ldots,C-1}$, then we train an ensemble of $C-1$ base-classifiers as follows:

* A 1<sup>st</sup> base-classifier is trained to distinguish the single class $0$ from the collection of classes $1,2,\ldots,C-1$ together. That is, all instances from class $0$ are assigned a target $y=0$ and all instances from classes $1,2,\ldots,C-1$ are assigned a target $y=1$ when training this base-classifier (thus forming a binary classification problem).

* A 2<sup>nd</sup> base-classifier is trained to distinguish the classes $0$ and $1$ together from the other classes $2,3,\ldots,C-1$ collectively. That is, all instances from classes $0,1$ are assigned a target $y=0$ and all instances from classes $2,3,\ldots,C-1$ are assigned a target $y=1$ when training this base-classifier (again forming a binary classification problem).

* etc.

* A $(C-1)$<sup>th</sup> base-classifier is trained to distinguish the classes $0,1,\ldots,C-2$ together from the single class $C-1$. That is, all instances from classes $0,1,\ldots,C-2$ are assigned a target $y=0$ and all instances from class $C-1$ alone are assigned a target $y=1$ when training this base-classifier (once more forming a binary classification problem).

Below, this is illustrated for $C=6$ classes labeled $0,1,2,3,4,5$.

<img src="pictures/ordinal_ensemble_classifier.png" width="40%" />

Once trained, all $C-1$ base-classifiers can be applied to a new test instance. The final ordinal label is then derived from those assigned by the base-classifiers by evaluating how many classifiers predict a class label from their "low" or "high" group. This can be achieved by simply summing the labels that are assigned by the base-classifiers. For example, if an instance from the actual class $0$ is correctly classified by all base-classifiers, then all base-classifiers will predict $\hat{y}=0$ because class 0 is always in the lower group of class labels. The sum of their assigned labels then — correctly — equals $0$.

Note that this approach only allows *one* ordinal class label to be assigned (i.e., the `predict` method can be implemented); it does *not* straightforwardly allow meaningful probabilities to be assigned to *all* classes (i.e., the `predict_proba` method cannot be implemented).

**Assignment**

* Write your own `sklearn`-compatible classifier class called `OrdinalEnsembleClassifier` that implements the above ensemble method. It should at least have `__init__`, `fit`, and `predict` methods. The initializer should have an `estimator` parameter that allows the user to supply a base-classifier instance; if necessary, consult `sklearn`'s documentation of the `OneVsOneClassifier` or `OneVsRestClassifier` classes that have an identical parameter. The following code fragment illustrates an example of the intended usage:

```python
base_clf = DummyClassifier()
meta_clf = OrdinalEnsembleClassifier(estimator=base_clf)
```

* Train your `OrdinalEnsembleClassifier` on the complete dataset to predict the Fetal state class code (NSP) from the 21 available numeric features; omit the categorical feature CLASS with the FHR pattern class code (A to J). Choose what you deem to be an appropriate base-classifier for your ensemble model and briefly motivate that choice. Mind the order of the N/S/P class labels in the target variable (since alphanumerically $\text{N} < \text{P}  < \text{S}$, but here $\text{N} < \text{S}  < \text{P}$).

* Report the model's achieved confusion matrix and accuracy based on resubstitution of the dataset. Based on the nature of the errors that the model makes, is the performance satisfactory?

In [ ]:
# YOUR ANSWER GOES HERE

<a id="4" href="#0" style="text-align: right; display: block;">Back to top</a>

## ❹ Part IV: Optimized predictive pipeline <small>[30 pts]</small>

Hoodbhoy et al. published an open-access short paper in which they compared various methods to classify the current dataset, except that they did not include the FHR pattern class code (A to J). In summary, the machine learning algorithms used in this study were a multilayer perceptron, support vector machines with linear and radial basis function kernels, k‐nearest neighbors, XGBoost and AdaBoost classifiers, random forest, logistic regression, Gaussian Naïve Bayes, and a decision tree. k‐Fold Cross Validation and augmentation were used to determine overall accuracy as well as precision, recall, and F1-score per class.

(Reference: Z. Hoodbhoy et al., "Use of Machine Learning Algorithms for Prediction of Fetal Risk using Cardiotocographic Data", *Int J Appl Basic Med Res*, 2019, 9(4), p.226-230; doi: [10.4103/ijabmr.IJABMR_370_18](https://journals.lww.com/ijab/fulltext/2019/09040/use_of_machine_learning_algorithms_for_prediction.9.aspx))

On the training data, the best models achieved high precision and recall (>96% and >99%, respectively) to predict the suspect (S) and pathological (P) state. However, on the test data sensitivity dropped to 92% to predict a pathological (P) fetal state and to 73% for the suspect (S) state. When comparing accuracy between training and test data, the best accuracy on the test data was 93%, and overall the below results were obtained (Fig. 1 in the paper).

<img src="pictures/Original.01634247-201909040-00009.F1-9.jpeg" width="40%" />

**Assignment**

* Develop an optimized analysis pipeline that, given the features of an unlabeled instance, is able to predict as reliably as possible the Fetal state class code (NSP) from the 21 other features, excluding the FHR pattern class code (CLASS).
  * Consider whether the dataset is sufficiently balanced, and if necessary take precautions to deal with any issues this may involve.
  * Compare at least five different classification methods, and choose the best.
  * Insofar as applicable and useful, perform some (k-fold) cross-validated hyperparameter optimization and/or include feature selection.

* Use proper cross-validation in order to be able to give a comprehensive unbiased final evaluation of your model based on independent test data that includes at the very least the precision and recall for the suspect (S) and pathologic (P) classes, as well as the overall accuracy. Do your own final results more or less match those by Hoodbhoy et al.?

For all major steps in your development process, explain the choices that you make and why you made them. For all relevant results that you obtain, provide an interpretation in your own words and draw evidence-based conclusions.

In [ ]:
# YOUR ANSWER GOES HERE

***

<a id="X" href="#0" style="text-align: right; display: block;">Back to top</a>

<div class="alert alert-info">
    
# ℹ️

**Note:** After finishing,

1. evaluate the notebook by means of the menu option `Kernel` > `Restart & Run All` and check that your notebook runs without errors;
2. save the evaluated notebook using the menu option `File` > `Save and Checkpoint`;
3. export your notebook as an *.html file using the menu option `File` > `Download as` > `HTML (.html)`;
4. place all your files in a folder and zip them (i.e. your completed notebook in *.ipynb and *.html form, plus any of your own required *.py modules; do **not** include original data files or any own modules that you did not actually use);
5. submit your work by attaching the compressed zip-file in BrightSpace, and submitting it;
6. if in doubt, you may check with the exam supervisor that your submission was successfully received;
7. you are free to leave!

</div>
    
*Success!*